## Basic

In [5]:
import os
from pydub import AudioSegment
import librosa
import torch
from scipy.spatial.distance import cosine
from speechbrain.pretrained import SpeakerRecognition
from sklearn.preprocessing import normalize


def convert_m4a_to_wav(audio_path):
    """
    Converts an .m4a file to .wav format.
    """
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Audio file {audio_path} does not exist.")
    
    if audio_path.endswith(".m4a"):
        converted_path = audio_path.replace(".m4a", ".wav")
        audio = AudioSegment.from_file(audio_path, format="m4a")
        audio.export(converted_path, format="wav")
        print(f"Converted {audio_path} to {converted_path}")
        return converted_path
    return audio_path

def process_audio(audio_path):
    """
    Preprocesses audio by loading and converting if necessary.
    Returns the audio signal and sampling rate.
    """
    # Convert .m4a to .wav if needed
    audio_path = convert_m4a_to_wav(audio_path)
    
    # Load audio using librosa
    signal, sr = librosa.load(audio_path, sr=16000, mono=True)
    print(f"Loaded audio: {audio_path}, Sample Rate: {sr}")
    return signal


def generate_embedding(signal, model):
    """
    Generates an embedding for the given audio signal using the pre-trained model.
    """
    embedding = model.encode_batch(torch.tensor([signal]))
    return embedding.squeeze().tolist()


def register_user(audio_path, model, embedding_storage_path):
    """
    Registers a user's voice by generating and saving an embedding.
    """
    # Process the audio file
    signal = process_audio(audio_path)
    
    # Generate embedding
    embedding = generate_embedding(signal, model)
    
    # Save the embedding
    torch.save(embedding, embedding_storage_path)
    print(f"User registered successfully. Embedding saved at {embedding_storage_path}.")


def authenticate_user(input_audio, embedding_storage_path, model, threshold=0.85):
    """
    Authenticates a user by comparing the input audio's embedding with a stored embedding.
    """
    # Check if the stored embedding exists
    if not os.path.exists(embedding_storage_path):
        raise FileNotFoundError("Stored embedding not found. Please register a user first.")
    
    # Load the stored embedding
    stored_embedding = torch.load(embedding_storage_path)
    
    # Process the input audio
    signal = process_audio(input_audio)
    
    # Generate embedding for the input audio
    input_embedding = generate_embedding(signal, model)
    
    # Compare embeddings using cosine similarity
    similarity_score = 1 - cosine(input_embedding, stored_embedding)
    print(f"Similarity Score: {similarity_score}")

    l2_input = normalize(input_embedding.reshape(1, -1)).flatten()
    l2_stored = normalize(stored_embedding.reshape(1, -1)).flatten()

    similarity_score = 1 - cosine(l2_input, l2_stored)
    print(f"Similarity Score L2: {similarity_score}")
    
    # Authenticate based on threshold
    if similarity_score > threshold:
        return "Authentication Successful"
    else:
        return "Authentication Failed"


# Main Logic
if __name__ == "__main__":
    registration_audio_path = "../Data/AUDIO_Test2.m4a"  # Replace with your registration audio file
    authentication_audio_path = "../Data/AUDIO_Test3.m4a"  # Replace with your authentication audio file
    embedding_storage_path = "user_embedding.pth"  # Path to save or load the stored embedding
    
    # Load the pre-trained speaker verification model
    model = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="tmp")
    
    # Perform Registration
    print("Registering User...")
    register_user(registration_audio_path, model, embedding_storage_path)
    
    # Perform Authentication
    print("Authenticating User...")
    result = authenticate_user(authentication_audio_path, embedding_storage_path, model)
    print(result)


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Registering User...
Converted ../Data/AUDIO_Test2.m4a to ../Data/AUDIO_Test2.wav
Loaded audio: ../Data/AUDIO_Test2.wav, Sample Rate: 16000
User registered successfully. Embedding saved at user_embedding.pth.
Authenticating User...
Converted ../Data/AUDIO_Test3.m4a to ../Data/AUDIO_Test3.wav
Loaded audio: ../Data/AUDIO_Test3.wav, Sample Rate: 16000
Similarity Score: 0.6590941888320668


AttributeError: 'list' object has no attribute 'reshape'

## Noise Reduction and Voice activity detection

In [25]:
import os
import librosa
import numpy as np
import torch
from scipy.io.wavfile import write
from scipy.signal import wiener
from scipy.spatial.distance import cosine
from speechbrain.pretrained import SpeakerRecognition


# Step 1: File Conversion
def convert_m4a_to_wav(audio_path):
    """
    Converts an .m4a file to .wav format.
    """
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Audio file {audio_path} does not exist.")
    
    if audio_path.endswith(".m4a"):
        converted_path = audio_path.replace(".m4a", ".wav")
        audio = AudioSegment.from_file(audio_path, format="m4a")
        audio.export(converted_path, format="wav")
        print(f"Converted {audio_path} to {converted_path}")
        return converted_path
    return audio_path


# Step 2: Load and Resample
def load_audio(audio_path, target_sr=16000):
    """
    Loads audio and resamples to a target sample rate.
    """
    signal, sr = librosa.load(audio_path, sr=None, mono=True)
    if sr != target_sr:
        signal = librosa.resample(signal, orig_sr=sr, target_sr=target_sr)
    print(f"Loaded audio: {audio_path} at {target_sr} Hz.")
    return signal


# Step 3: Noise Reduction
def noise_reduction(signal, sr, n_fft=2048, hop_length=512):
    """
    Reduces noise using spectral subtraction and Wiener filtering.
    """
    # Spectrogram-based noise reduction
    stft = librosa.stft(signal, n_fft=n_fft, hop_length=hop_length)
    magnitude, phase = librosa.magphase(stft)
    noise_est = np.mean(magnitude[:, :int(sr * 0.5)], axis=1, keepdims=True)
    denoised_mag = np.maximum(magnitude - noise_est, 0)
    denoised_signal = librosa.istft(denoised_mag * phase, hop_length=hop_length)
    
    # Wiener filtering for residual noise
    denoised_signal = wiener(denoised_signal)
    print("Noise reduction applied.")
    return denoised_signal


# Step 4: Voice Activity Detection
def voice_activity_detection(signal, sr, frame_length=512, hop_length=256, energy_threshold=0.02):
    """
    Performs voice activity detection to isolate speech segments.
    """
    energy = np.array([
        np.sum(np.abs(signal[i:i + frame_length]) ** 2)
        for i in range(0, len(signal), hop_length)
    ])
    mask = energy > (energy_threshold * np.max(energy))
    vad_signal = np.concatenate([signal[i * hop_length:(i + 1) * hop_length] for i in range(len(mask)) if mask[i]])
    print("Voice activity detection applied.")
    return vad_signal


# Step 5: Embedding Generation
def generate_embedding(signal, model):
    """
    Generates a speaker embedding using a pre-trained model.
    """
    # Convert signal to PyTorch tensor
    signal_tensor = torch.tensor(signal).float().unsqueeze(0)
    embedding = model.encode_batch(signal_tensor).squeeze().tolist()
    print("Speaker embedding generated.")
    return embedding


# Step 6: Register User
def register_user(audio_path, model, embedding_storage_path):
    """
    Registers a user's voice by saving their embedding.
    """
    # Full preprocessing pipeline
    signal = load_audio(audio_path)
    # signal = noise_reduction(signal, sr=16000)
    # signal = voice_activity_detection(signal, sr=16000)
    
    # Generate and save embedding
    embedding = generate_embedding(signal, model)
    torch.save(embedding, embedding_storage_path)
    print(f"User registered successfully. Embedding saved at {embedding_storage_path}.")


# Step 7: Authenticate User
def authenticate_user(input_audio, embedding_storage_path, model, threshold=0.85):
    """
    Authenticates a user by comparing input audio embedding with a stored embedding.
    """
    if not os.path.exists(embedding_storage_path):
        raise FileNotFoundError("No registered embedding found. Please register a user first.")
    
    # Load stored embedding
    stored_embedding = torch.load(embedding_storage_path)
    
    # Full preprocessing pipeline
    signal = load_audio(input_audio)
    # signal = noise_reduction(signal, sr=16000)
    # signal = voice_activity_detection(signal, sr=16000)
    
    # Generate input embedding
    input_embedding = generate_embedding(signal, model)
    
    # Cosine similarity
    similarity_score = 1 - cosine(input_embedding, stored_embedding)
    print(f"Similarity Score: {similarity_score}")
    
    # Authentication result
    return "Authentication Successful" if similarity_score > threshold else "Authentication Failed"


# Main Functionality
if __name__ == "__main__":
    # File paths
    registration_audio_path = "../Data/Ravi.wav"
    authentication_audio_path = "../Data/RaviTest.wav"
    embedding_storage_path = "user_embedding.pth"
    
    # Load pre-trained model
    model = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="tmp")
    
    # Registration
    print("Registering user...")
    register_user(registration_audio_path, model, embedding_storage_path)
    
    # Authentication
    print("Authenticating user...")
    result = authenticate_user(authentication_audio_path, embedding_storage_path, model)
    print(result)


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Registering user...
Loaded audio: ../Data/Ravi.wav at 16000 Hz.
Speaker embedding generated.
User registered successfully. Embedding saved at user_embedding.pth.
Authenticating user...
Loaded audio: ../Data/RaviTest.wav at 16000 Hz.
Speaker embedding generated.
Similarity Score: 0.6170178661730302
Authentication Failed


## Transcription added

In [26]:
import os
from pydub import AudioSegment
import librosa
import numpy as np
import torch
from scipy.spatial.distance import cosine
from speechbrain.pretrained import SpeakerRecognition
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import soundfile as sf

def convert_m4a_to_wav(audio_path):
    """
    Converts an .m4a file to .wav format.
    """
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Audio file {audio_path} does not exist.")
    
    if audio_path.endswith(".m4a"):
        converted_path = audio_path.replace(".m4a", ".wav")
        audio = AudioSegment.from_file(audio_path, format="m4a")
        audio.export(converted_path, format="wav")
        print(f"Converted {audio_path} to {converted_path}")
        return converted_path
    return audio_path

def remove_silence(signal, sr, top_db=30):
    """
    Removes silence from the audio signal.
    """
    non_silent_intervals = librosa.effects.split(signal, top_db=top_db)
    non_silent_signal = np.concatenate([signal[start:end] for start, end in non_silent_intervals])
    print(f"Removed silence. Original length: {len(signal)}, New length: {len(non_silent_signal)}")
    return non_silent_signal

def process_audio(audio_path):
    """
    Preprocesses audio by loading, converting if necessary, resampling, and removing silence.
    Returns the audio signal and sampling rate.
    """
    # Convert .m4a to .wav if needed
    audio_path = convert_m4a_to_wav(audio_path)

    # Load audio using librosa and resample to 16,000 Hz
    signal, sr = librosa.load(audio_path, sr=16000, mono=True)
    print(f"Loaded and resampled audio: {audio_path}, Sample Rate: {sr}")

    # Remove silence from the audio
    # signal = remove_silence(signal, sr)
    return signal

def generate_embedding(signal, model):
    """
    Generates an embedding for the given audio signal using the pre-trained model.
    """
    embedding = model.encode_batch(torch.tensor([signal]))
    print(f"Embedding dimension: {embedding.shape[-1]}")

    return embedding.squeeze().tolist()

def transcribe_audio(audio_path, processor, model):
    """
    Transcribes the audio to text using a pre-trained Wav2Vec2 model.
    """
    # Convert .m4a to .wav if needed and load the audio
    audio_path = convert_m4a_to_wav(audio_path)

    # Resample the audio if needed
    signal, sr = librosa.load(audio_path, sr=16000, mono=True)
    print(f"Transcribing audio: Resampled to {sr} Hz")

    # Preprocess audio for the model
    input_values = processor(signal, sampling_rate=sr, return_tensors="pt", padding=True).input_values
    logits = model(input_values).logits

    # Decode transcription
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0]
    print(f"Transcription: {transcription}")
    return transcription

def register_user(audio_path, model, embedding_storage_path, processor, stt_model, transcription_storage_path):
    """
    Registers a user's voice by generating and saving an embedding and transcribing the speech.
    """
    # Process the audio file
    signal = process_audio(audio_path)
    
    # Generate embedding
    embedding = generate_embedding(signal, model)
    
    # Save the embedding
    torch.save(embedding, embedding_storage_path)
    print(f"User registered successfully. Embedding saved at {embedding_storage_path}.")

    # Transcribe the audio
    transcription = transcribe_audio(audio_path, processor, stt_model)
    with open(transcription_storage_path, "w") as f:
        f.write(transcription)
    print(f"Transcription saved at {transcription_storage_path}.")

def authenticate_user(input_audio, embedding_storage_path, model, processor, stt_model, transcription_storage_path, threshold=0.85):
    """
    Authenticates a user by comparing the input audio's embedding with a stored embedding
    and matching the transcribed speech.
    """
    # Check if the stored embedding exists
    if not os.path.exists(embedding_storage_path):
        raise FileNotFoundError("Stored embedding not found. Please register a user first.")
    
    # Load the stored embedding
    stored_embedding = torch.load(embedding_storage_path)
    
    # Process the input audio
    signal = process_audio(input_audio)
    
    # Generate embedding for the input audio
    input_embedding = generate_embedding(signal, model)
    
    # Compare embeddings using cosine similarity
    similarity_score = 1 - cosine(input_embedding, stored_embedding)
    print(f"Similarity Score: {similarity_score}")

    # Authenticate based on threshold
    embedding_match = similarity_score > threshold

    # Transcribe the input audio
    input_transcription = transcribe_audio(input_audio, processor, stt_model)

    # Load the stored transcription
    with open(transcription_storage_path, "r") as f:
        stored_transcription = f.read().strip()
    
    # Compare transcriptions
    transcription_match = input_transcription.strip().lower() == stored_transcription.lower()

    if embedding_match and transcription_match:
        return "Authentication Successful"
    elif not embedding_match:
        return "Authentication Failed: Voice does not match"
    else:
        return "Authentication Failed: Speech does not match"

# Main Logic
if __name__ == "__main__":
    registration_audio_path = "../Data/Ravi.wav"
    authentication_audio_path = "../Data/RaviTest.wav"
    embedding_storage_path = "user_embedding.pth"  # Path to save or load the stored embedding
    transcription_storage_path = "user_transcription.txt"  # Path to save or load the stored transcription
    

    # Load the pre-trained speaker verification model
    speaker_model = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="tmp")

    # Load the pre-trained speech-to-text model
    processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
    stt_model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

    # Perform Registration
    print("Registering User...")
    register_user(registration_audio_path, speaker_model, embedding_storage_path, processor, stt_model, transcription_storage_path)

    # Perform Authentication
    print("Authenticating User...")
    result = authenticate_user(authentication_audio_path, embedding_storage_path, speaker_model, processor, stt_model, transcription_storage_path)
    print(result)


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
Some weights of Wav2Vec2ForCTC were not initial

Registering User...
Loaded and resampled audio: ../Data/Ravi.wav, Sample Rate: 16000
Embedding dimension: 192
User registered successfully. Embedding saved at user_embedding.pth.
Transcribing audio: Resampled to 16000 Hz
Transcription: I SHALL WAKE HYBAD
Transcription saved at user_transcription.txt.
Authenticating User...
Loaded and resampled audio: ../Data/RaviTest.wav, Sample Rate: 16000
Embedding dimension: 192
Similarity Score: 0.6170178661730302
Transcribing audio: Resampled to 16000 Hz
Transcription: ALOSHEWIK HALOVAT
Authentication Failed: Voice does not match


## Indian Accent 

In [20]:
import os
import torch
import librosa
import numpy as np
import noisereduce as nr
import webrtcvad
from pydub import AudioSegment
from difflib import SequenceMatcher
from scipy.spatial.distance import cosine
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from speechbrain.pretrained import SpeakerRecognition

def convert_m4a_to_wav(audio_path):
    """Convert .m4a to .wav format with proper channel handling"""
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Audio file {audio_path} does not exist.")
    
    if audio_path.endswith(".m4a"):
        converted_path = audio_path.replace(".m4a", ".wav")
        audio = AudioSegment.from_file(audio_path, format="m4a")
        audio.set_channels(1).export(converted_path, format="wav")
        print(f"Converted {audio_path} to {converted_path}")
        return converted_path
    return audio_path

def audio_preprocessing_pipeline(audio_path):
    """Comprehensive audio preprocessing pipeline"""
    # Convert format and ensure mono channel
    audio_path = convert_m4a_to_wav(audio_path)
    
    # Load audio with librosa (forces mono, resampling to 16kHz)
    signal, sr = librosa.load(audio_path, sr=16000, mono=True)
    
    # Noise reduction
    signal = nr.reduce_noise(y=signal, sr=sr, stationary=True)
    
    # Voice activity detection
    vad = webrtcvad.Vad(3)
    frame_duration = 30  # ms
    frame_length = int(sr * frame_duration / 1000)
    
    # Convert to int16 PCM for VAD and ensure correct length
    signal_int16 = (signal * 32767).astype(np.int16)
    truncated_length = (len(signal_int16) // frame_length) * frame_length
    signal_truncated = signal_int16[:truncated_length]
    
    if truncated_length == 0:
        raise ValueError("Audio too short for VAD processing")
    
    # Split into exact frames
    frames = np.reshape(signal_truncated, (-1, frame_length))
    
    # Detect speech frames
    speech_frames = [frame for frame in frames if vad.is_speech(frame.tobytes(), sr)]
    
    if not speech_frames:
        raise ValueError("No speech detected in audio")
    
    # Convert back to float32 for audio processing
    speech_signal_int16 = np.concatenate(speech_frames)
    signal = speech_signal_int16.astype(np.float32) / 32767.0
    
    # Normalization and silence trimming
    signal = librosa.util.normalize(signal)
    signal, _ = librosa.effects.trim(signal, top_db=25)
    
    return signal, sr
def transcribe_audio(signal, sr, processor, model):
    """Transcribe audio using Whisper with Indian accent optimization"""
    inputs = processor(
        signal, 
        sampling_rate=sr, 
        return_tensors="pt",
        language="english",
        task="transcribe"
    )
    
    # For Indian accents, use more beam searches
    predicted_ids = model.generate(
        inputs.input_features,
        num_beams=5,
        language="en",
        task="transcribe"
    )
    
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    print("Transcription:", transcription)
    return transcription.lower().strip()

def text_similarity(text1, text2):
    """Calculate text similarity score with sequence matching"""
    return SequenceMatcher(None, text1, text2).ratio()

def register_user(audio_path, speaker_model, stt_processor, stt_model, storage_path):
    """Register user with voice embedding and speech content"""
    signal, _ = audio_preprocessing_pipeline(audio_path)
    
    # Store voice embedding
    embedding = speaker_model.encode_batch(torch.tensor([signal])).squeeze()
    torch.save(embedding, storage_path)
    
    # Store speech content
    transcription = transcribe_audio(signal, 16000, stt_processor, stt_model)
    with open(storage_path.replace(".pth", ".txt"), "w") as f:
        f.write(transcription)
    
    print(f"Registered: {transcription}")

def authenticate_user(audio_path, storage_path, speaker_model, stt_processor, stt_model, 
                     speaker_thresh=0.85, text_thresh=0.75):
    """Multi-factor authentication: voice + content"""
    # Load registered data
    if not os.path.exists(storage_path):
        raise FileNotFoundError("User not registered")
    
    reg_embedding = torch.load(storage_path)
    with open(storage_path.replace(".pth", ".txt")) as f:
        reg_text = f.read().lower().strip()
    
    # Process input audio
    signal, _ = audio_preprocessing_pipeline(audio_path)
    
    # Speaker verification
    input_embedding = speaker_model.encode_batch(torch.tensor([signal])).squeeze()
    speaker_score = 1 - cosine(input_embedding.numpy(), reg_embedding.numpy())
    
    # Content verification
    input_text = transcribe_audio(signal, 16000, stt_processor, stt_model)
    text_score = text_similarity(reg_text, input_text)
    
    print(f"Speaker Similarity: {speaker_score:.2f}, Content Match: {text_score:.2f}")
    
    if speaker_score >= speaker_thresh and text_score >= text_thresh:
        return "Authentication Successful"
    return "Authentication Failed"

if __name__ == "__main__":
    # Initialize models
    speaker_model = SpeakerRecognition.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb",
        savedir="pretrained_models/speaker"
    )
    
    stt_processor = WhisperProcessor.from_pretrained("openai/whisper-medium")
    stt_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium")
    
    # Example usage
    registration_path = "../Data/Paarth1.m4a"
    auth_path = "../Data/Paarth2.m4a"
    storage_path = "user_profile.pth"
    
    # Registration
    print("Registering user...")
    register_user(registration_path, speaker_model, stt_processor, stt_model, storage_path)
    
    # Authentication
    print("\nAuthenticating...")
    result = authenticate_user(auth_path, storage_path, speaker_model, stt_processor, stt_model)
    print(f"Result: {result}")

ModuleNotFoundError: No module named 'noisereduce'

In [19]:
import os
import numpy as np
from scipy.spatial.distance import cosine
from sklearn.preprocessing import normalize, minmax_scale, StandardScaler
from pydub import AudioSegment
import librosa
import torch
from speechbrain.pretrained import SpeakerRecognition


def convert_m4a_to_wav(audio_path):
    """
    Converts an .m4a file to .wav format.
    """
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Audio file {audio_path} does not exist.")
    
    if audio_path.endswith(".m4a"):
        converted_path = audio_path.replace(".m4a", ".wav")
        audio = AudioSegment.from_file(audio_path, format="m4a")
        audio.export(converted_path, format="wav")
        print(f"Converted {audio_path} to {converted_path}")
        return converted_path
    return audio_path


def process_audio(audio_path):
    """
    Preprocesses audio by loading and converting if necessary.
    Returns the audio signal and sampling rate.
    """
    # Convert .m4a to .wav if needed
    audio_path = convert_m4a_to_wav(audio_path)
    
    # Load audio using librosa
    signal, sr = librosa.load(audio_path, sr=16000, mono=True)
    print(f"Loaded audio: {audio_path}, Sample Rate: {sr}")
    return signal


def generate_embedding(signal, model):
    """
    Generates an embedding for the given audio signal using the pre-trained model.
    """
    embedding = model.encode_batch(torch.tensor([signal]))
    return embedding.squeeze().numpy()


def normalize_embeddings(embedding, method="l2"):
    """
    Normalizes the embedding using the specified method.
    """
    if method == "l2":
        return normalize(embedding.reshape(1, -1)).flatten()  # L2 normalization
    elif method == "minmax":
        return minmax_scale(embedding.reshape(1, -1)).flatten()  # Min-Max normalization
    elif method == "zscore":
        return StandardScaler().fit_transform(embedding.reshape(1, -1)).flatten()  # Z-Score normalization
    elif method == "none":
        return embedding  # No normalization
    else:
        raise ValueError(f"Unknown normalization method: {method}")


def compare_normalization_methods(audio_path, model):
    """
    Compares the effect of different normalization methods on cosine distance.
    """
    # Process the audio file
    signal = process_audio(audio_path)
    
    # Generate the raw embedding
    raw_embedding = generate_embedding(signal, model)
    
    # Normalize the embedding using different methods
    normalization_methods = ["none", "l2", "minmax", "zscore"]
    results = {}
    
    for method in normalization_methods:
        normalized_embedding = normalize_embeddings(raw_embedding, method)
        distance = cosine(raw_embedding, normalized_embedding)
        results[method] = distance
    
    return results


# Main Logic
if __name__ == "__main__":
    audio_path = "../Data/Paarth2.m4a"  # Replace with your audio file
    
    # Load the pre-trained speaker verification model
    model = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="tmp")
    
    # Compare normalization methods
    results = compare_normalization_methods(audio_path, model)
    
    # Print results
    print("Cosine Distance Between Raw and Normalized Embeddings:")
    for method, distance in results.items():
        print(f"{method.upper()} Normalization: {distance:.4f}")
    
    # Recommend the best normalization method
    best_method = min(results, key=results.get)
    print(f"\nRecommendation: Use {best_method.upper()} normalization for minimal cosine distance.")

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Converted ../Data/Paarth2.m4a to ../Data/Paarth2.wav
Loaded audio: ../Data/Paarth2.wav, Sample Rate: 16000
Cosine Distance Between Raw and Normalized Embeddings:
NONE Normalization: 0.0000
L2 Normalization: 0.0000
MINMAX Normalization: nan
ZSCORE Normalization: nan

Recommendation: Use NONE normalization for minimal cosine distance.


/home/llm/Voice_Recognition/venv/lib/python3.10/site-packages/scipy/spatial/distance.py:685: RuntimeWarning: invalid value encountered in scalar divide
  dist = 1.0 - uv / math.sqrt(uu * vv)
/home/llm/Voice_Recognition/venv/lib/python3.10/site-packages/scipy/spatial/distance.py:685: RuntimeWarning: invalid value encountered in scalar divide
  dist = 1.0 - uv / math.sqrt(uu * vv)


## Using PCA and L2

In [27]:
import os
import librosa
import numpy as np
import torch
from scipy.io.wavfile import write
from scipy.signal import wiener
from scipy.spatial.distance import cosine
from speechbrain.pretrained import SpeakerRecognition
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize


# Step 1: File Conversion
def convert_m4a_to_wav(audio_path):
    """
    Converts an .m4a file to .wav format.
    """
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Audio file {audio_path} does not exist.")
    
    if audio_path.endswith(".m4a"):
        converted_path = audio_path.replace(".m4a", ".wav")
        audio = AudioSegment.from_file(audio_path, format="m4a")
        audio.export(converted_path, format="wav")
        print(f"Converted {audio_path} to {converted_path}")
        return converted_path
    return audio_path


# Step 2: Load and Resample
def load_audio(audio_path, target_sr=16000):
    """
    Loads audio and resamples to a target sample rate.
    """
    signal, sr = librosa.load(audio_path, sr=None, mono=True)
    if sr != target_sr:
        signal = librosa.resample(signal, orig_sr=sr, target_sr=target_sr)
    print(f"Loaded audio: {audio_path} at {target_sr} Hz.")
    return signal


# Step 3: Noise Reduction
def noise_reduction(signal, sr, n_fft=2048, hop_length=512):
    """
    Reduces noise using spectral subtraction and Wiener filtering.
    """
    # Spectrogram-based noise reduction
    stft = librosa.stft(signal, n_fft=n_fft, hop_length=hop_length)
    magnitude, phase = librosa.magphase(stft)
    noise_est = np.mean(magnitude[:, :int(sr * 0.5)], axis=1, keepdims=True)
    denoised_mag = np.maximum(magnitude - noise_est, 0)
    denoised_signal = librosa.istft(denoised_mag * phase, hop_length=hop_length)
    
    # Wiener filtering for residual noise
    denoised_signal = wiener(denoised_signal)
    print("Noise reduction applied.")
    return denoised_signal


# Step 4: Voice Activity Detection
def voice_activity_detection(signal, sr, frame_length=512, hop_length=256, energy_threshold=0.02):
    """
    Performs voice activity detection to isolate speech segments.
    """
    energy = np.array([
        np.sum(np.abs(signal[i:i + frame_length]) ** 2)
        for i in range(0, len(signal), hop_length)
    ])
    mask = energy > (energy_threshold * np.max(energy))
    vad_signal = np.concatenate([signal[i * hop_length:(i + 1) * hop_length] for i in range(len(mask)) if mask[i]])
    print("Voice activity detection applied.")
    return vad_signal


# Step 5: Embedding Generation with L2 Normalization
def generate_embedding(signal, model):
    """
    Generates a speaker embedding using a pre-trained model and normalizes the embedding.
    """
    # Convert signal to PyTorch tensor
    signal_tensor = torch.tensor(signal).float().unsqueeze(0)
    embedding = model.encode_batch(signal_tensor).squeeze().tolist()
    
    # Apply L2 normalization to the embedding
    embedding = np.array(embedding)
    embedding = normalize(embedding.reshape(1, -1), norm='l2').flatten()  # L2 Normalize
    print("Speaker embedding generated and normalized.")
    return embedding


# Step 6: PCA Post-Processing (Fit PCA on a set of embeddings)
def fit_pca(embeddings, n_components=50):
    """
    Fits PCA on a collection of embeddings and reduces dimensionality.
    """
    pca = PCA(n_components=n_components)
    embeddings_reduced = pca.fit_transform(embeddings)
    print(f"PCA fitted, reducing embeddings to {n_components} dimensions.")
    return pca

# Step 7: Register User (Save embedding using numpy)
def register_user(audio_path, model, embedding_storage_path, embeddings, pca=None):
    """
    Registers a user's voice by saving their embedding with optional PCA post-processing.
    """
    # Full preprocessing pipeline
    signal = load_audio(audio_path)
    # signal = noise_reduction(signal, sr=16000)
    # signal = voice_activity_detection(signal, sr=16000)
    
    # Generate and save embedding
    embedding = generate_embedding(signal, model)
    
    # Apply PCA if needed (only if PCA is already fitted)
    if pca:
        embedding = pca.transform([embedding])
    
    # Save the embedding using numpy
    np.save(embedding_storage_path, embedding)
    print(f"User registered successfully. Embedding saved at {embedding_storage_path}.")
    
    # Collect the embedding for PCA fitting
    embeddings.append(embedding)
    
    # If enough embeddings are collected, fit PCA
    if len(embeddings) > 4:  # For example, require at least 5 embeddings
        pca = fit_pca(embeddings)  # Fit PCA with collected embeddings
        print("PCA fitted on collected embeddings.")
    
    return pca

# Step 8: Authenticate User (Load embedding using numpy)
def authenticate_user(input_audio, embedding_storage_path, model, pca=None, threshold=0.85):
    """
    Authenticates a user by comparing input audio embedding with a stored embedding.
    """
    if not os.path.exists(embedding_storage_path):
        raise FileNotFoundError("No registered embedding found. Please register a user first.")
    
    # Load stored embedding using numpy
    stored_embedding = np.load(embedding_storage_path)
    
    # Full preprocessing pipeline
    signal = load_audio(input_audio)
    # signal = noise_reduction(signal, sr=16000)
    # signal = voice_activity_detection(signal, sr=16000)
    
    # Generate input embedding
    input_embedding = generate_embedding(signal, model)
    
    # Apply PCA if needed
    if pca:
        input_embedding = pca.transform([input_embedding])
    
    # Cosine similarity
    similarity_score = 1 - cosine(input_embedding, stored_embedding)
    print(f"Similarity Score: {similarity_score}")
    
    # Authentication result
    return "Authentication Successful" if similarity_score > threshold else "Authentication Failed"

# Main Functionality
if __name__ == "__main__":
    # File paths
    registration_audio_path = "../Data/Ravi.wav"
    authentication_audio_path = "../Data/RaviTest.wav"
    embedding_storage_path = "user_embedding.npy"  # Changed to .npy for NumPy
    
    # Load pre-trained model
    model = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="tmp")
    
    # Optionally initialize PCA for dimensionality reduction
    embeddings = []  # Collect a list of embeddings for PCA fitting
    pca = None
    
    # Register user
    print("Registering user...")
    pca = register_user(registration_audio_path, model, embedding_storage_path, embeddings, pca)
    
    # Authenticate user
    print("Authenticating user...")
    result = authenticate_user(authentication_audio_path, embedding_storage_path, model, pca)
    print(result)


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Registering user...
Loaded audio: ../Data/Ravi.wav at 16000 Hz.
Speaker embedding generated and normalized.
User registered successfully. Embedding saved at user_embedding.npy.
Authenticating user...
Loaded audio: ../Data/RaviTest.wav at 16000 Hz.
Speaker embedding generated and normalized.
Similarity Score: 0.6170178661730304
Authentication Failed


## Combining 

In [23]:
import os
import numpy as np
import librosa
import torch
from scipy.spatial.distance import cosine
from speechbrain.pretrained import SpeakerRecognition
from sklearn.decomposition import PCA
from scipy.signal import wiener
from pydub import AudioSegment


# Helper function for file conversion (m4a to wav)
def convert_m4a_to_wav(audio_path):
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Audio file {audio_path} does not exist.")
    
    if audio_path.endswith(".m4a"):
        converted_path = audio_path.replace(".m4a", ".wav")
        audio = AudioSegment.from_file(audio_path, format="m4a")
        audio.export(converted_path, format="wav")
        return converted_path
    return audio_path


# Step 1: Preprocessing
def preprocess_audio(audio_path, target_sr=16000):
    """
    Full preprocessing pipeline including noise reduction and voice activity detection.
    """
    # Convert m4a to wav if necessary
    audio_path = convert_m4a_to_wav(audio_path)
    
    # Load audio
    signal, sr = librosa.load(audio_path, sr=target_sr, mono=True)
    
    # Noise Reduction (Wiener filtering for residual noise)
    signal = wiener(signal)
    
    # Voice Activity Detection (VAD)
    energy = np.array([np.sum(np.abs(signal[i:i + 512])**2) for i in range(0, len(signal), 256)])
    mask = energy > 0.02 * np.max(energy)
    vad_signal = np.concatenate([signal[i * 256:(i + 1) * 256] for i in range(len(mask)) if mask[i]])
    
    return vad_signal


# Step 2: Embedding Generation with Normalization
def generate_embedding_with_normalization(signal, model):
    """
    Generate embedding from audio signal and apply L2 normalization.
    """
    # Convert to tensor
    signal_tensor = torch.tensor(signal).float().unsqueeze(0)
    
    # Generate embedding
    embedding = model.encode_batch(signal_tensor).squeeze().tolist()
    
    # Apply L2 normalization to embedding
    embedding = np.array(embedding)
    embedding = embedding / np.linalg.norm(embedding)  # L2 Normalization
    
    return embedding


# Step 3: PCA for Dimensionality Reduction (Optional)
def apply_pca_to_embeddings(embeddings, n_components=50):
    """
    Apply PCA to reduce dimensionality and ensure embeddings are in the same space.
    """
    pca = PCA(n_components=n_components)
    embeddings_reduced = pca.fit_transform(embeddings)
    return pca, embeddings_reduced


# Step 4: Data Augmentation Techniques
def augment_audio(signal, sr, augment_type='noise', intensity=0.005):
    """
    Apply data augmentation techniques such as pitch shifting or noise injection.
    """
    if augment_type == 'noise':
        noise = np.random.normal(0, intensity, len(signal))
        augmented_signal = signal + noise
    elif augment_type == 'pitch':
        augmented_signal = librosa.effects.pitch_shift(signal, sr, n_steps=4)  # Pitch shift by 4 semitones
    elif augment_type == 'speed':
        augmented_signal = librosa.effects.time_stretch(signal, 0.8)  # Speed perturbation
    else:
        augmented_signal = signal
    
    return augmented_signal


# Step 5: Authentication with Cosine Similarity
def authenticate_user(input_audio, stored_embedding, model, pca=None, threshold=0.85):
    """
    Authenticate a user based on cosine similarity.
    """
    # Preprocess the input audio
    signal = preprocess_audio(input_audio)
    
    # Generate embedding with normalization
    input_embedding = generate_embedding_with_normalization(signal, model)
    
    # Apply PCA if available
    if pca:
        input_embedding = pca.transform([input_embedding])
    
    # Cosine similarity
    similarity_score = 1 - cosine(input_embedding, stored_embedding)
    print(f"Similarity Score: {similarity_score}")
    
    # Authentication result
    return "Authentication Successful" if similarity_score > threshold else "Authentication Failed"


# Step 6: User Registration
def register_user(audio_path, model, embedding_storage_path, embeddings, pca=None):
    """
    Register a new user's voice by saving their embedding.
    """
    # Preprocess the audio
    signal = preprocess_audio(audio_path)
    
    # Generate embedding with normalization
    embedding = generate_embedding_with_normalization(signal, model)
    
    # Apply PCA if needed
    if pca:
        embedding = pca.transform([embedding])
    
    # Save embedding to disk
    np.save(embedding_storage_path, embedding)
    print(f"User registered successfully. Embedding saved at {embedding_storage_path}.")
    
    # Collect the embedding for PCA fitting
    embeddings.append(embedding)
    
    # Fit PCA if we have enough embeddings (e.g., 5 or more)
    if len(embeddings) >= 5:
        pca, embeddings_reduced = apply_pca_to_embeddings(embeddings)
        print("PCA fitted on collected embeddings.")
    
    return pca


# Main function to tie everything together
if __name__ == "__main__":
    # File paths
    registration_audio_path = "../Data/Ravi.wav"
    authentication_audio_path = "../Data/RaviTest.wav"
    embedding_storage_path = "user_embedding.npy"
    
    # Load the pre-trained model
    model = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="tmp")
    
    # Store embeddings for PCA fitting
    embeddings = []  # Collect a list of embeddings
    pca = None
    
    # Register user (First, apply augmentation if needed)
    print("Registering user...")
    pca = register_user(registration_audio_path, model, embedding_storage_path, embeddings, pca)
    
    # Authenticate user
    print("Authenticating user...")
    result = authenticate_user(authentication_audio_path, np.load(embedding_storage_path), model, pca)
    print(result)


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Registering user...
User registered successfully. Embedding saved at user_embedding.npy.
Authenticating user...
Similarity Score: 0.5474703800032352
Authentication Failed


## Improved Augmentation

In [28]:
import os
import numpy as np
import librosa
import torch
from scipy.spatial.distance import cosine
from speechbrain.pretrained import SpeakerRecognition
from pydub import AudioSegment


# Helper function for file conversion (m4a to wav)
def convert_m4a_to_wav(audio_path):
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Audio file {audio_path} does not exist.")
    
    if audio_path.endswith(".m4a"):
        converted_path = audio_path.replace(".m4a", ".wav")
        audio = AudioSegment.from_file(audio_path, format="m4a")
        audio.export(converted_path, format="wav")
        return converted_path
    return audio_path


# Step 1: Preprocessing (Keep signal clean)
def preprocess_audio(audio_path, target_sr=16000):
    """
    Full preprocessing pipeline including noise reduction and voice activity detection.
    """
    # Convert m4a to wav if necessary
    audio_path = convert_m4a_to_wav(audio_path)
    
    # Load audio
    signal, sr = librosa.load(audio_path, sr=target_sr, mono=True)
    
    # Noise Reduction (Wiener filtering for residual noise)
    signal = librosa.effects.preemphasis(signal)
    
    # Optional: Apply some simple filtering to remove frequencies outside of human speech range (300-3400 Hz)
    # signal = librosa.effects.trim(signal)  # This trims leading/trailing silence
    
    return signal


# Step 2: Embedding Generation without Normalization
def generate_embedding_without_normalization(signal, model):
    """
    Generate embedding from audio signal without normalization.
    """
    # Convert to tensor
    signal_tensor = torch.tensor(signal).float().unsqueeze(0)
    
    # Generate embedding
    embedding = model.encode_batch(signal_tensor).squeeze().tolist()
    
    return embedding


# Step 3: Data Augmentation Techniques (with more varied types)
def augment_audio(signal, sr, augment_type='noise', intensity=0.005):
    """
    Apply data augmentation techniques such as pitch shifting or noise injection.
    """
    if augment_type == 'noise':
        noise = np.random.normal(0, intensity, len(signal))
        augmented_signal = signal + noise
    elif augment_type == 'pitch':
        augmented_signal = librosa.effects.pitch_shift(signal, sr=sr, n_steps=4)  # Pitch shift by 4 semitones
    elif augment_type == 'speed':
        augmented_signal = librosa.effects.time_stretch(signal, 0.8)  # Speed perturbation
    elif augment_type == 'reverb':
        augmented_signal = signal + np.random.normal(0, 0.1, len(signal))  # Reverb effect
    else:
        augmented_signal = signal
    
    return augmented_signal


# Step 4: Authentication with Cosine Similarity
def authenticate_user(input_audio, stored_embedding, model, threshold=0.85):
    """
    Authenticate a user based on cosine similarity.
    """
    # Preprocess the input audio
    signal = preprocess_audio(input_audio)
    
    # Apply data augmentation to input signal (for robustness)
    signal = augment_audio(signal, 16000, augment_type='noise', intensity=0.01)
    
    # Generate embedding without normalization
    input_embedding = generate_embedding_without_normalization(signal, model)
    
    # Cosine similarity
    similarity_score = 1 - cosine(input_embedding, stored_embedding)
    print(f"Similarity Score: {similarity_score}")
    
    # Authentication result
    return "Authentication Successful" if similarity_score > threshold else "Authentication Failed"


# Step 5: User Registration
def register_user(audio_path, model, embedding_storage_path):
    """
    Register a new user's voice by saving their embedding.
    """
    # Preprocess the audio
    signal = preprocess_audio(audio_path)
    
    # Apply augmentation to the signal
    augmented_signal = augment_audio(signal, 16000, augment_type='pitch')
    
    # Generate embedding without normalization
    embedding = generate_embedding_without_normalization(augmented_signal, model)
    
    # Save embedding to disk
    np.save(embedding_storage_path, embedding)
    print(f"User registered successfully. Embedding saved at {embedding_storage_path}.")
    
    return embedding


# Main function to tie everything together
if __name__ == "__main__":
    # File paths
    registration_audio_path = "../Data/Ravi.wav"
    authentication_audio_path = "../Data/RaviTest.wav"
    embedding_storage_path = "user_embedding.npy"
    
    # Load the pre-trained model
    model = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="tmp")
    
    # Register user
    print("Registering user...")
    registered_embedding = register_user(registration_audio_path, model, embedding_storage_path)
    
    # Authenticate user
    print("Authenticating user...")
    result = authenticate_user(authentication_audio_path, np.load(embedding_storage_path), model)
    print(result)


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Registering user...
User registered successfully. Embedding saved at user_embedding.npy.
Authenticating user...
Similarity Score: 0.019752856667415775
Authentication Failed
